# Building REDCap training pairs from the PhenX Toolkit

Every PhenX protocol ships the same variables in two parallel files:

- the **PhenX Data Dictionary** (`DD_NNNNNN_finalized.csv`) with `VARNAME, VARDESC, TYPE, VALUES`
- the **REDCap instrument** (inside `PXNNNNNN.zip`) with the usual REDCap columns
  (`Variable / Field Name, Form Name, Field Type, Field Label, Choices`)

So for each variable we can build a pair:

- **anchor** = how the variable looks in REDCap
- **positive** = how the same variable looks in the PhenX dictionary

We line the two files up per variable, then write out the two paragraphs. The REDCap file
always starts with a `record_id` row that the dictionary doesn't have, so we drop it first.

In [1]:
import os
import re
import io
import zipfile
import subprocess

import pandas as pd

DATA_DIR = "phenx_data"
os.makedirs(DATA_DIR, exist_ok=True)

DD_URL     = "https://www.phenxtoolkit.org/toolkit_content/documents/data_dictionary/ALL_DD_CSV_Files.zip"
REDCAP_URL = "https://www.phenxtoolkit.org/toolkit_content/documents/data_dictionary/ALL_REDCap_ZIP_Files.zip"

DD_ZIP     = os.path.join(DATA_DIR, "ALL_DD_CSV_Files.zip")
REDCAP_ZIP = os.path.join(DATA_DIR, "ALL_REDCap_ZIP_Files.zip")

# grab both archives once; skip if we already have them
for url, path in [(DD_URL, DD_ZIP), (REDCAP_URL, REDCAP_ZIP)]:
    if os.path.exists(path):
        print("already downloaded:", path)
    else:
        print("downloading", url, "...")
        subprocess.run(["wget", "-q", "-O", path, url], check=True)
    print("   size: %.1f MB" % (os.path.getsize(path) / 1e6))

already downloaded: phenx_data/ALL_DD_CSV_Files.zip
   size: 1.7 MB
already downloaded: phenx_data/ALL_REDCap_ZIP_Files.zip
   size: 26.8 MB


## Index both archives by protocol number

The dictionary files are named `DD_851001_finalized.csv` and the REDCap ones `PX851001.zip`,
so the six digits in the middle are what ties a pair together.

In [2]:
dd_archive = zipfile.ZipFile(DD_ZIP)
rc_archive = zipfile.ZipFile(REDCAP_ZIP)

dd_map = {re.search(r"DD_(\d+)_", n).group(1): n
          for n in dd_archive.namelist() if re.search(r"DD_(\d+)_", n)}
rc_map = {re.search(r"PX(\d+)\.zip", n).group(1): n
          for n in rc_archive.namelist() if re.search(r"PX(\d+)\.zip", n)}

protocols = sorted(set(dd_map) & set(rc_map))

print("dictionary protocols :", len(dd_map))
print("redcap protocols     :", len(rc_map))
print("in both (usable)     :", len(protocols))

dictionary protocols : 1185
redcap protocols     : 983
in both (usable)     : 973


## A couple of small helpers

Some of the dictionary CSVs are saved with a UTF-8 BOM, which sticks a few junk characters on the
first column name (`VARNAME` becomes `ï»¿VARNAME`) and then nothing matches. Reading with
`utf-8-sig` strips the BOM; if a file isn't UTF-8 we fall back to `latin-1`.

In [3]:
def read_csv_bytes(raw):
    """Read a CSV from raw bytes, handling the BOM some PhenX files have."""
    try:
        return pd.read_csv(io.BytesIO(raw), encoding="utf-8-sig", dtype=str).fillna("")
    except Exception:
        return pd.read_csv(io.BytesIO(raw), encoding="latin-1", dtype=str).fillna("")


def collect_values(row, start_idx):
    """The dictionary spreads a variable's answer options across VALUES + the trailing
    unnamed columns, so we walk from VALUES to the end and keep the non-empty cells.

    Example: a row with VALUES='1 = Yes', Unnamed:14='2 = No', Unnamed:15=''
             -> '1 = Yes; 2 = No'
    """
    out = []
    for v in row.iloc[start_idx:]:
        s = str(v).strip()
        if s and s.lower() != "nan":
            out.append(s)
    return "; ".join(out)


def strip_form_prefix(form):
    """Drop the pxNNNNNN_ prefix from a form name.

    Example: 'px851001_phenx_transfusion_reaction' -> 'phenx_transfusion_reaction'
    """
    return re.sub(r"^px\d+_", "", str(form))


def dd_key(varname):
    """Turn a dictionary VARNAME into the join key: drop the PXNNNNNN_ prefix, lowercase.

    Example: 'PX851001_Transfusion_Reaction_Participant_Age' -> 'transfusion_reaction_participant_age'
    which is exactly the REDCap 'Variable / Field Name'.
    """
    s = str(varname)
    return s.split("_", 1)[1].lower() if "_" in s else s.lower()


def clean_varname(varname):
    """Readable variable name for the positive side: drop the prefix, underscores to spaces.

    Example: 'PX851001_Transfusion_Reaction_Participant_Age' -> 'Transfusion Reaction Participant Age'
    """
    s = str(varname)
    return s.split("_", 1)[1].replace("_", " ") if "_" in s else s

## Build the two paragraphs

`anchor` uses REDCap's four required columns (Field Name, Form Name, Field Type, Field Label)
plus Choices, and `positive` uses the dictionary's VARNAME / VARDESC / TYPE / VALUES.
Choices and VALUES are only added when they actually have something in them.

In [4]:
def build_anchor(row):
    parts = [
        "Field Name: "  + row["Variable / Field Name"],
        "Form Name: "   + strip_form_prefix(row["Form Name"]),
        "Field Type: "  + row["Field Type"],
        "Field Label: " + row["Field Label"],
    ]
    choices = str(row["Choices, Calculations, OR Slider Labels"]).strip()
    if choices and choices.lower() != "nan":
        choices = re.sub(r"\s*\|\s*", "; ", choices)   # only change: option separator | -> "; "
        parts.append("Choices: " + choices)
    return " | ".join(parts)


def build_positive(row):
    parts = [
        "VARNAME: "  + row["varname_clean"],
        "VARDESC: "  + row["VARDESC"],
        "TYPE: "     + row["TYPE"],
    ]
    values = str(row["values_joined"]).strip()
    if values:
        parts.append("VALUES: " + values)
    return " | ".join(parts)

## Process one protocol

Read both files, drop the `record_id` row, line the variables up by name (an inner join, so
order doesn't matter and we only keep variables that exist on both sides), then build the pairs.

In [5]:
def process_protocol(num):
    # --- REDCap instrument ---
    inner = zipfile.ZipFile(io.BytesIO(rc_archive.read(rc_map[num])))
    csv_name = [n for n in inner.namelist() if n.lower().endswith(".csv")][0]
    rc = read_csv_bytes(inner.read(csv_name))
    if "Variable / Field Name" not in rc.columns:
        return None
    rc = rc[~rc["Variable / Field Name"].str.endswith("_record_id")].copy()
    rc["key"] = rc["Variable / Field Name"].str.lower()

    # --- PhenX dictionary ---
    dd = read_csv_bytes(dd_archive.read(dd_map[num]))
    if "VARNAME" not in dd.columns or "VALUES" not in dd.columns:
        return None
    values_start = list(dd.columns).index("VALUES")
    dd["values_joined"] = dd.apply(lambda r: collect_values(r, values_start), axis=1)
    dd["key"] = dd["VARNAME"].apply(dd_key)
    dd["varname_clean"] = dd["VARNAME"].apply(clean_varname)

    # --- line them up by variable name ---
    merged = rc.merge(dd, on="key", how="inner")
    if merged.empty:
        return None

    merged["variable_para"] = merged.apply(build_anchor, axis=1)
    merged["alternate_variable_para"] = merged.apply(build_positive, axis=1)
    merged["protocol"] = num
    return merged[["protocol", "variable_para", "alternate_variable_para"]]

## Quick sanity check on one protocol before running everything

851001 is the transfusion-reaction protocol. Let's make sure a pair looks right.

In [6]:
sample = process_protocol("851001")
print("pairs in protocol 851001:", len(sample))
print()
print("ANCHOR  :", sample.iloc[3]["variable_para"])
print("POSITIVE:", sample.iloc[3]["alternate_variable_para"])

pairs in protocol 851001: 209

ANCHOR  : Field Name: transfusion_reaction_participant_facility | Form Name: phenx_transfusion_reaction | Field Type: radio | Field Label: Did the transfusion occur at your facility? | Choices: UNDEFINED_CODE, Yes; UNDEFINED_CODE_1, No
POSITIVE: VARNAME: Transfusion Reaction Participant Facility | VARDESC: Did the transfusion occur at your facility? | TYPE: enumerated | VALUES: Yes; No


## Run it over every protocol

We also keep track of the ones that don't produce any pairs so we can see why.

In [7]:
frames = []
unmatched = []

for num in protocols:
    try:
        pairs = process_protocol(num)
    except Exception as e:
        unmatched.append((num, "error: " + str(e)[:40]))
        continue
    if pairs is None or len(pairs) == 0:
        unmatched.append((num, "no matching variables"))
    else:
        frames.append(pairs)

phenx_pairs = pd.concat(frames, ignore_index=True)

print("protocols that produced pairs :", len(frames))
print("protocols with no pairs       :", len(unmatched))
print("total REDCap pairs            :", len(phenx_pairs))

protocols that produced pairs : 672
protocols with no pairs       : 301
total REDCap pairs            : 21541


## Why do some protocols not match?

Most of the unmatched ones aren't a naming problem we can fix -- their dictionary simply isn't
laid out the same way. In those files the `VARNAME` column holds the *question text* or an
*answer option* instead of the variable name, so there's nothing to line up against the REDCap
field names. A few examples:

In [8]:
# show a handful of unmatched protocols and what their keys look like on each side
shown = 0
for num, reason in unmatched:
    if shown >= 6:
        break
    try:
        inner = zipfile.ZipFile(io.BytesIO(rc_archive.read(rc_map[num])))
        cn = [n for n in inner.namelist() if n.lower().endswith(".csv")][0]
        rc = read_csv_bytes(inner.read(cn))
        dd = read_csv_bytes(dd_archive.read(dd_map[num]))
        if "Variable / Field Name" not in rc.columns or "VARNAME" not in dd.columns:
            continue
        rc = rc[~rc["Variable / Field Name"].str.endswith("_record_id")]
        rc_key = rc["Variable / Field Name"].str.lower().iloc[0]
        dd_k = dd_key(dd["VARNAME"].iloc[0])
        print(f"proto {num}:")
        print(f"    RC key: {rc_key}   <- REDCap uses the variable name")
        print(f"    DD key: {dd_k}   <- dictionary has something else here")
        shown += 1
    except Exception:
        continue

proto 010102:
    RC key: age_birthdate   <- REDCap uses the variable name
    DD key: what is your birthdate?   <- dictionary has something else here
proto 010802:
    RC key: address_physical   <- REDCap uses the variable name
    DD key: please tell me your complete physical street address.  {#} {direction} {street name} {street/road/avenue} {direction} {#} {po box} {rural route #} {rural routebox} {city} {state} {zip}   <- dictionary has something else here
proto 011002:
    RC key: educational_attainment_individual_highest_grade   <- REDCap uses the variable name
    DD key: 7 = 7th grade   <- dictionary has something else here
proto 011102:
    RC key: annual_family_income_household_members   <- REDCap uses the variable name
    DD key:    <- dictionary has something else here
proto 011901:
    RC key: race_ethnicity_hispanic   <- REDCap uses the variable name
    DD key:    <- dictionary has something else here
proto 012001:
    RC key: income_all   <- REDCap uses the variable n

## Look at the pairs we actually care about

The benchmark leans heavily on demographics, so let's eyeball a real pair for each of
race, ethnicity, address, employment, education, and income.

In [9]:
for keyword in ["race", "ethnic", "address", "employ", "educ", "income"]:
    hits = phenx_pairs[phenx_pairs["variable_para"].str.contains(keyword, case=False, na=False)]
    print("=" * 100)
    print(f"{keyword.upper()}  ({len(hits)} matching pairs)")
    if len(hits):
        print("ANCHOR  :", hits.iloc[0]["variable_para"])
        print("POSITIVE:", hits.iloc[0]["alternate_variable_para"])
    print()

RACE  (237 matching pairs)
ANCHOR  : Field Name: race_self_identification | Form Name: phenx_race | Field Type: radio | Field Label: What race or races do you consider yourself to be? Please select one or more. | Choices: 1, AMERICAN INDIAN OR ALASKA NATIVE; 2, ASIAN [GO TO Q2]; 3, BLACK OR AFRICAN AMERICAN; 4, NATIVE HAWAIIAN OR PACIFIC ISLANDER [GO TO Q3]; 5, WHITE; 6, OTHER; 99, DON'T KNOW; 77, REFUSED
POSITIVE: VARNAME: Race Self Identification | VARDESC: What race or races do you consider yourself to be? Please select one or more. | TYPE: encoded values | VALUES: 1 = AMERICAN INDIAN OR ALASKA NATIVE; 2 = ASIAN [GO TO Q2]; 3 = BLACK OR AFRICAN AMERICAN; 4 = NATIVE HAWAIIAN OR PACIFIC ISLANDER [GO TO Q3]; 5 = WHITE; 6 = OTHER; 99 = DONT KNOW; 77 = REFUSED

ETHNIC  (234 matching pairs)
ANCHOR  : Field Name: ethnicity_self_identification | Form Name: phenx_ethnicity | Field Type: radio | Field Label: Do you consider yourself to be Hispanic, Latino, or of Spanish origin? | Choices: 1,

ADDRESS  (75 matching pairs)
ANCHOR  : Field Name: current_street_number | Form Name: phenx_current_address | Field Type: text | Field Label: I would like to verify your address. Please give me your complete address. STREET Number
POSITIVE: VARNAME: Current Street Number | VARDESC: I would like to verify your address.  Please give me your complete address.  Number | TYPE: integer

EMPLOY  (129 matching pairs)
ANCHOR  : Field Name: current_employment_status | Form Name: phenx_current_employment_status | Field Type: radio | Field Label: We would like to know about what you do --are you working now, looking for work, retired, keeping house, a student, or what? | Choices: 1, WORKING NOW; 2, ONLY TEMPORARILY LAID OFF, SICK LEAVE OR MATERNITY LEAVE; 3, LOOKING FOR WORK, UNEMPLOYED; 4, RETIRED; 5, DISABLED, PERMANENTLY OR TEMPORARILY; 6, KEEPING HOUSE; 7, STUDENT; 8, OTHER (SPECIFY):
POSITIVE: VARNAME: Current Employment Status | VARDESC: We would like to know about what you do --are you work

EDUC  (174 matching pairs)
ANCHOR  : Field Name: current_educational_attainment | Form Name: phenx_current_educational_attainment | Field Type: radio | Field Label: What is the highest grade or level of school you have completed or the highest degree you have received? [HAND CARD DMQ1. READ HAND CARD CATEGORIES IF NECESSARY. ENTER HIGHEST LEVEL OF SCHOOL.] | Choices: 0, NEVER ATTENDED/KINDERGARTEN ONLY; 1, 1ST GRADE; 2, 2ND GRADE; 3, 3RD GRADE; 4, 4TH GRADE; 5, 5TH GRADE; 6, 6TH GRADE; 7, 7TH GRADE; 8, 8TH GRADE; 9, 9TH GRADE; 10, 10TH GRADE; 11, 11TH GRADE; 12, 12TH GRADE, NO DIPLOMA; 13, HIGH SCHOOL GRADUATE; 14, GED OR EQUIVALENT; 15, SOME COLLEGE, NO DEGREE; 16, ASSOCIATE DEGREE: OCCUPATIONAL, TECHNICAL, OR VOCATIONAL PROGRAM; 17, ASSOCIATE DEGREE: ACADEMIC PROGRAM; 18, BACHELOR'S DEGREE (EXAMPLE: BA, AB, BS, BBA); 19, MASTER'S DEGREE (EXAMPLE: MA, MS, MEng, MEd, MBA); 20, PROFESSIONAL SCHOOL DEGREE (EXAMPLE: MD, DDS, DVM, JD); 21, DOCTORAL DEGREE (EXAMPLE: PhD, EdD); 77, REFUSED; 

INCOME  (29 matching pairs)
ANCHOR  : Field Name: annual_family_income | Form Name: phenx_annual_family_income | Field Type: text | Field Label: What was your best estimate of the total income of all family members from all sources, before taxes, in last year?
POSITIVE: VARNAME: Annual Family Income | VARDESC: What was your best estimate of the total income of all family members from all sources, before taxes, in last year? | TYPE: integer | VALUES: 0..999994



## Save

Same two column names as the NCIt no-dup file (`variable_para` / `alternate_variable_para`),
so this drops straight into the training data alongside the NCIt pairs.

In [10]:
out_path = os.path.join(DATA_DIR, "phenx_redcap_pairs.csv")
phenx_pairs[["variable_para", "alternate_variable_para"]].to_csv(out_path, index=False)
print("saved", len(phenx_pairs), "pairs to", out_path)

saved 21541 pairs to phenx_data/phenx_redcap_pairs.csv
